In [60]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
from sklearn.metrics import r2_score
import seaborn as sns
from datetime import datetime
from tabulate import tabulate
import pandas as pd

sns.set_theme(style='ticks', context='paper')

In [46]:
# Constants
ENERGY_CONSUMPTION = "Energy Consumption (exc. PUE)"
ENERGY_CONSUMPTION_PUE = "Energy Consumption (inc. PUE)"
MEMORY_CONSUMPTION = "Memory Energy Consumption (exc. PUE)"
MEMORY_CONSUMPTION_PUE = "Memory Energy Consumption (inc. PUE)"
CARBON_EMISSIONS = "Carbon Emissions"

In [57]:
# Trace File Paths List
workflows = ['chipseq', 'mag', 'montage', 'nanoseq', 'rangeland', 'rnaseq', 'sarek']
regions = ['Great Britain']#, 'Germany', 'California', 'Texas', 'South Africa', 'Tokyo', 'New South Wales']
short_regions = ['gb']#, 'de', 'ca', 'tx', 'zaf', 'tyo', 'nsw']
months = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
short_months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'] 
month_dates = ['08012024-13012024', '12022024-17022024', '11032024-16032024',  
    '08042024-13042024', '13052024-18052024', '10062024-15062024',  
    '08072024-13072024', '12082024-17082024', '09092024-14092024',  
    '14102024-19102024', '11112024-16112024', '09122024-14122024']
region_folders = [f'../data/results/interrupt-wf-shifting/out/{region}/' for region in short_regions]
windows = [6, 12, 24, 48, 96]

In [113]:
def get_max_overhead(overheads):
    oh_1, _ = overheads[0].split('|')
    oh_2, _ = overheads[1].split('|')
    oh_3, _ = overheads[2].split('|')

    return max([float(oh_1), float(oh_2), float(oh_3)])
    

def parse_ts_summary_file(file):
    with open(file, 'r') as f:
        lines = f.readlines()[1:]

    props = {}

    for line in lines:
        parts = line.split(',')
        workflow = parts[0].split('-')[0]
        footprint = float(parts[1])
        makespan = float(parts[2])
        entry_6h = parts[3].split(':')
        entry_12h = parts[4].split(':')
        entry_24h = parts[5].split(':')
        entry_48h = parts[6].split(':')
        entry_96h = parts[7].split(':')

        if len(props) == 0:
            props = {
                "FOOTPRINT": [footprint],
                "MAKESPAN": [makespan],
                "6H_FOOTPRINT": [float(entry_6h[1])],
                "6H_REDUCTION": [float(entry_6h[0][:-1])],
                "6H_OVERHEAD": [entry_6h[2]],
                "12H_FOOTPRINT": [float(entry_12h[1])],
                "12H_REDUCTION": [float(entry_12h[0][:-1])],
                "12H_OVERHEAD": [entry_12h[2]],
                "24H_FOOTPRINT": [float(entry_24h[1])],
                "24H_REDUCTION": [float(entry_24h[0][:-1])],
                "24H_OVERHEAD": [entry_24h[2]],
                "48H_FOOTPRINT": [float(entry_48h[1])],
                "48H_REDUCTION": [float(entry_48h[0][:-1])],
                "48H_OVERHEAD": [entry_48h[2]],
                "96H_FOOTPRINT": [float(entry_96h[1])],
                "96H_REDUCTION": [float(entry_96h[0][:-1])],
                "96H_OVERHEAD": [entry_96h[2]]
            }
        else: 
            props["FOOTPRINT"].append(footprint)
            props["MAKESPAN"].append(makespan)
            props["6H_FOOTPRINT"].append(float(entry_6h[1]))
            props["6H_REDUCTION"].append(float(entry_6h[0][:-1]))
            props["6H_OVERHEAD"].append(entry_6h[2])
            props["12H_FOOTPRINT"].append(float(entry_12h[1]))
            props["12H_REDUCTION"].append(float(entry_12h[0][:-1]))
            props["12H_OVERHEAD"].append(entry_12h[2])
            props["24H_FOOTPRINT"].append(float(entry_24h[1]))
            props["24H_REDUCTION"].append(float(entry_24h[0][:-1]))
            props["24H_OVERHEAD"].append(entry_24h[2])
            props["48H_FOOTPRINT"].append(float(entry_48h[1]))
            props["48H_REDUCTION"].append(float(entry_48h[0][:-1]))
            props["48H_OVERHEAD"].append(entry_48h[2])
            props["96H_FOOTPRINT"].append(float(entry_96h[1]))
            props["96H_REDUCTION"].append(float(entry_96h[0][:-1]))
            props["96H_OVERHEAD"].append(entry_96h[2])

    data = {}
    wf_makespan = sum(props["MAKESPAN"]) / 3
    footprint_6h = sum(props["6H_FOOTPRINT"]) / 3
    reduction_6h = sum(props["6H_REDUCTION"]) / 3
    max_overhead_6h = get_max_overhead(props["6H_OVERHEAD"])
    footprint_12h = sum(props["12H_FOOTPRINT"]) / 3
    reduction_12h = sum(props["12H_REDUCTION"]) / 3

    if reduction_12h < reduction_6h:
        reduction_12h = reduction_6h

    max_overhead_12h = get_max_overhead(props["12H_OVERHEAD"])
    footprint_24h = sum(props["24H_FOOTPRINT"]) / 3
    reduction_24h = sum(props["24H_REDUCTION"]) / 3

    if reduction_24h < reduction_12h:
        reduction_24h = reduction_12h

    max_overhead_24h = get_max_overhead(props["24H_OVERHEAD"])
    footprint_48h = sum(props["48H_FOOTPRINT"]) / 3
    reduction_48h = sum(props["48H_REDUCTION"]) / 3

    if reduction_48h < reduction_24h:
        reduction_48h = reduction_24h

    max_overhead_48h = get_max_overhead(props["48H_OVERHEAD"])
    footprint_96h = sum(props["96H_FOOTPRINT"]) / 3
    reduction_96h = sum(props["96H_REDUCTION"]) / 3

    if reduction_96h < reduction_48h:
        reduction_96h = reduction_48h

    max_overhead_96h = get_max_overhead(props["6H_OVERHEAD"])

    data = {
        "FOOTPRINT": sum(props["FOOTPRINT"]) / 3,
        "MAKESPAN": wf_makespan,
        "6H_REDUCTION": reduction_6h,
        "6H_OVERHEAD":  max_overhead_6h,
        "6H_FOOTPRINT": footprint_6h,
        "12H_REDUCTION":  reduction_12h,
        "12H_OVERHEAD":  max_overhead_12h,
        "12H_FOOTPRINT": footprint_12h,
        "24H_REDUCTION":  reduction_24h,
        "24H_OVERHEAD":  max_overhead_24h,
        "24H_FOOTPRINT": footprint_24h,
        "48H_REDUCTION":  reduction_48h,
        "48H_OVERHEAD":  max_overhead_48h,
        "48H_FOOTPRINT": footprint_48h,
        "96H_REDUCTION":  reduction_96h,
        "96H_OVERHEAD":  max_overhead_96h,
        "96H_FOOTPRINT": footprint_96h,
    }

    return data
        

In [49]:
def write_to_file(outfile, data, headers):
    with open(f'../data/results/interrupt-wf-shifting/out/{outfile}', 'w') as f:
        f.write(','.join(headers) + '\n')

        for row in data:
            f.write(','.join(row) + '\n')

# purpose illustrating impact of pausing on overall footprint
- here, we consider the overhead in seconds, and convert this to a time in h
- we use this time to then account for memory energy consumption, where this considers all of the memory in the nodes used
- e.g. 256GB on all 8 nodes, assuming that data is stored here for time spent waiting (the overhead)
- this used the constant 0.392W/GB, and average CI data from 2023 UK -- 215gCO2e
- here, we see that there is little change to the overall reduction in footprint, with minimal emissions added on
- of course, you could then dive further into static power consumed by nodes waiting -- but when we pause, we assume that it could be throttled down / used by someone else, so we would not be responsible for those emissions
- if we focus on the impact of the shifting experiment -- only for our workflow execution, stored memory has little impact 
- don't consider moving it to an external system -- find references for whether this would incur a significant cost or not?

In [ ]:
# Interrupting Footprints - Average CI
headers = ["Region", "Month", "Workflow", "Original Footprint (gCO2e)", "Makespan (s)", 
           "Reduction 6h (%)", "Overhead 6h", "FP 6h",
           "Reduction 12h (%)", "Overhead 12h", "FP 12h",
           "Reduction 24h (%)", "Overhead 24h", "FP 24h",
           "Reduction 48h (%)", "Overhead 48h", "FP 48h",
           "Reduction 96h (%)","Overhead 96h", "FP 96h"]
data_avg_tbl = []
data_avg_dct = {}

for region, region_base in zip(short_regions, region_folders):
    if region not in data_avg_dct:
        data_avg_dct[region] = {}
    for month in short_months:
        for workflow in workflows:
            if workflow not in data_avg_dct[region]:
                data_avg_dct[region][workflow] = {}
            if month not in data_avg_dct[region][workflow]:
                data_avg_dct[region][workflow][month] = {} 
            temp = parse_ts_summary_file(f'{region_base}{workflow}-{month}-avg-ts.csv')
            span = temp["MAKESPAN"]
            data_avg_tbl.append([region, month, workflow,f'{temp["FOOTPRINT"]:.2f}', f'{span:.0f}', 
                 f"{temp['6H_REDUCTION']:.2f}", f"{temp['6H_OVERHEAD']:.2f}", f"{temp['6H_FOOTPRINT']:.2f}",
                 f"{temp['12H_REDUCTION']:.2f}", f"{temp['12H_OVERHEAD']:.2f}", f"{temp['12H_FOOTPRINT']:.2f}",
                 f"{temp['24H_REDUCTION']:.2f}", f"{temp['24H_OVERHEAD']:.2f}", f"{temp['24H_FOOTPRINT']:.2f}",
                 f"{temp['48H_REDUCTION']:.2f}", f"{temp['48H_OVERHEAD']:.2f}", f"{temp['48H_FOOTPRINT']:.2f}",
                 f"{temp['96H_REDUCTION']:.2f}", f"{temp['96H_OVERHEAD']:.2f}", f"{temp['96H_FOOTPRINT']:.2f}"])

            data_avg_dct[region][workflow][month][6] = temp['6H_REDUCTION']
            data_avg_dct[region][workflow][month][12] = temp['12H_REDUCTION']
            data_avg_dct[region][workflow][month][24] = temp['24H_REDUCTION']
            data_avg_dct[region][workflow][month][48] = temp['48H_REDUCTION']
            data_avg_dct[region][workflow][month][96] = temp['96H_REDUCTION']


# Accounting Example (marginal doesn't make much sense -- this is for the discussion)
limited_df = pd.DataFrame.from_records(data_avg_tbl, columns=headers)
limited_df = limited_df.drop(columns=['Region', 'Month'])
limited_df['OrFP'] = limited_df['Original Footprint (gCO2e)'].astype(float)
limited_df['span_s'] = limited_df['Makespan (s)'].astype(float)
limited_df['red_6h_per'] = limited_df['Reduction 6h (%)'].astype(float)
limited_df['red_12h_per'] = limited_df['Reduction 12h (%)'].astype(float)
limited_df['red_24h_per'] = limited_df['Reduction 24h (%)'].astype(float)
limited_df['red_48h_per'] = limited_df['Reduction 48h (%)'].astype(float)
limited_df['red_96h_per'] = limited_df['Reduction 96h (%)'].astype(float)
limited_df['oh_6h_s'] = limited_df['Overhead 6h'].astype(float)
limited_df['oh_12h_s'] = limited_df['Overhead 12h'].astype(float)
limited_df['oh_24h_s'] = limited_df['Overhead 24h'].astype(float)
limited_df['oh_48h_s'] = limited_df['Overhead 48h'].astype(float)
limited_df['oh_96h_s'] = limited_df['Overhead 96h'].astype(float)
limited_df['fp_6h'] = limited_df['FP 6h'].astype(float)
limited_df['fp_12h'] = limited_df['FP 12h'].astype(float)
limited_df['fp_24h'] = limited_df['FP 24h'].astype(float)
limited_df['fp_48h'] = limited_df['FP 48h'].astype(float)
limited_df['fp_96h'] = limited_df['FP 96h'].astype(float)
avg = limited_df.groupby(['Workflow']).mean(numeric_only=True).reset_index()

for workflow, node_mem, nodes in zip(workflows, [128, 256, 128, 32, 256, 128, 128], [8, 8, 8, 1, 8, 8, 8]):
    row = avg[avg['Workflow'] == workflow].iloc[0].to_dict()
    mem_oh_24h_energy = node_mem * nodes * 0.392 * (row['oh_24h_s'] / 3600) * 0.001  # convert from W to kW
    mem_oh_24h_ems = mem_oh_24h_energy * 215
    tot_24h_ems = mem_oh_24h_ems + row['fp_24h']
    mem_oh_96h_energy = node_mem * nodes * 0.392 * (row['oh_96h_s'] / 3600) * 0.001  # convert from W to kW
    mem_oh_96h_ems = mem_oh_96h_energy * 215
    tot_96h_ems = mem_oh_96h_ems + row['fp_96h']

    print(f'{workflow} Original Footprint {row['OrFP']:.2f}')
    print(f'24H Overhead {mem_oh_24h_ems:.2f}gCO2e Total {tot_24h_ems:.2f}gCO2e Reduction {(row['OrFP'] - tot_24h_ems) /  row['OrFP'] * 100:.2f}% prev. {row['red_24h_per']:.2f}%')
    print(f'96h Overhead {mem_oh_96h_ems:.2f}gCO2e Total {tot_96h_ems:.2f}gCO2e Reduction {(row['OrFP'] - tot_96h_ems) / row['OrFP'] * 100:.2f}% prev. {row['red_96h_per']:.2f}%')
    print('\n')

    # re ichnos, National Grid Average CI in 2023 is 215gCO2e (rough estimation)


chipseq Original Footprint 759.50
24H Overhead 4.40gCO2e Total 450.53gCO2e Reduction 40.68% prev. 40.48%
96h Overhead 2.82gCO2e Total 361.24gCO2e Reduction 52.44% prev. 50.81%


mag Original Footprint 1574.40
24H Overhead 132.06gCO2e Total 1113.13gCO2e Reduction 29.30% prev. 36.82%
96h Overhead 126.27gCO2e Total 905.54gCO2e Reduction 42.48% prev. 48.55%


montage Original Footprint 37.94
24H Overhead 0.00gCO2e Total 25.12gCO2e Reduction 33.80% prev. 34.03%
96h Overhead 0.00gCO2e Total 18.85gCO2e Reduction 50.33% prev. 51.46%


nanoseq Original Footprint 40.22
24H Overhead 0.31gCO2e Total 25.34gCO2e Reduction 37.00% prev. 35.31%
96h Overhead 0.07gCO2e Total 20.35gCO2e Reduction 49.40% prev. 45.64%


rangeland Original Footprint 714.60
24H Overhead 9.19gCO2e Total 463.93gCO2e Reduction 35.08% prev. 33.69%
96h Overhead 4.59gCO2e Total 369.25gCO2e Reduction 48.33% prev. 44.58%


rnaseq Original Footprint 531.82
24H Overhead 11.45gCO2e Total 318.70gCO2e Reduction 40.07% prev. 41.90%
96h Ove

In [52]:
# Constants
regions = ['Great Britain', 'Germany', 'California', 'Texas', 'South Africa', 'Tokyo', 'New South Wales']
short_regions = ['gb', 'de', 'ca', 'tx', 'zaf', 'tyo', 'nsw']
months = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
short_months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'] 
month_dates = ['08012024-13012024', '12022024-17022024', '11032024-16032024',  
    '08042024-13042024', '13052024-18052024', '10062024-15062024',  
    '08072024-13072024', '12082024-17082024', '09092024-14092024',  
    '14102024-19102024', '11112024-16112024', '09122024-14122024']
region_folders = [f'../data/results/entire-wf-shifting/out-err/{region}/' for region in short_regions]
workflows=['chipseq', 'mag', 'montage', 'nanoseq', 'rangeland', 'rnaseq', 'sarek']

In [53]:
# Parse Explorer Summary (NEW)
def parse_explorer_summary(explorer_data):
    with open(explorer_data, 'r') as f:
        data = [line.strip().split(',') for line in f.readlines()]

    for row in data:
        if '~' in row[0]:
            shift_ms = row[0].split('~')[1][:-4]
            days = int(shift_ms.split('-')[0])
            hours = int(shift_ms.split('-')[1])
            shift_h = (days * 24) + hours
            row[0] = shift_h
        else:
            row[0] = 0

    data_d = {}

    for row in data:
        data_d[row[0]] = {'emissions': row[1]}

    return data_d


def get_minimum(data):
    minimum = float(data[0]['emissions'])
    shift = 0

    for key, entry in data.items():
        if float(entry['emissions']) < minimum:
            minimum = float(entry['emissions'])
            shift = key

    return (shift, minimum)


def get_reduction(original, new):
    orig = float(original)
    neww = float(new)
    return ((orig - neww) / abs(orig)) * 100


def overview(trace, print=False):
    data = parse_explorer_summary(trace)
    (min_shift, min_emissions) = get_minimum(data)

    if print:
        print(f"Original [0] CCF {data[0]['emissions']} gCO2e")  # original
        print(f"Minimum [{min_shift}] CCF {min_emissions} gCO2e")  # minimum
        print(f"Reduction {get_reduction(data[0]['emissions'], min_emissions):.2f}%")

    return (data, min_shift, min_emissions)


def average(one, two, three):
    return (float(one) + float(two) + float(three)) / 3

In [54]:
# Parse Readings for all readings, through the year
def parse_readings(window, marg='', err=''):
    region = 'gb'
    folder = f'../data/results/entire-wf-shifting/out-err/{region}/'

    if err == '':
        reduction_by_workflow = {}

        for i in range(0, 12):
            for workflow in workflows:
                path_one = f'{folder}explorer-{window}h-{workflow}-{short_months[i]}-1-{region}-{month_dates[i]}{err}{marg}~footprint.csv'
                path_two = f'{folder}explorer-{window}h-{workflow}-{short_months[i]}-2-{region}-{month_dates[i]}{err}{marg}~footprint.csv'
                path_three = f'{folder}explorer-{window}h-{workflow}-{short_months[i]}-3-{region}-{month_dates[i]}{err}{marg}~footprint.csv'

                (data_1, _, min_emissions_1) = overview(path_one)
                (data_2, _, min_emissions_2) = overview(path_two)
                (data_3, _, min_emissions_3) = overview(path_three)
                avg_orig_ems = average(data_1[0]['emissions'], data_2[0]['emissions'], data_3[0]['emissions'])
                avg_min_ems = average(min_emissions_1, min_emissions_2, min_emissions_3)
                avg_reduction = get_reduction(avg_orig_ems, avg_min_ems)

                if workflow in reduction_by_workflow:
                    reduction_by_workflow[workflow].append(avg_reduction)
                else:
                    reduction_by_workflow[workflow] = [avg_reduction]

        return reduction_by_workflow

    data = {}

    for rep in range(1, 6):
        reduction_by_workflow = {}

        for i in range(0, 12):
            for workflow in workflows:
                path_one = f'{folder}explorer-{window}h-{workflow}-{short_months[i]}-1-{region}-{month_dates[i]}{err}{marg}~footprint.csv'
                path_two = f'{folder}explorer-{window}h-{workflow}-{short_months[i]}-2-{region}-{month_dates[i]}{err}{marg}~footprint.csv'
                path_three = f'{folder}explorer-{window}h-{workflow}-{short_months[i]}-3-{region}-{month_dates[i]}{err}{marg}~footprint.csv'

                (data_1, _, min_emissions_1) = overview(path_one)
                (data_2, _, min_emissions_2) = overview(path_two)
                (data_3, _, min_emissions_3) = overview(path_three)
                avg_orig_ems = average(data_1[0]['emissions'], data_2[0]['emissions'], data_3[0]['emissions'])
                avg_min_ems = average(min_emissions_1, min_emissions_2, min_emissions_3)
                avg_reduction = get_reduction(avg_orig_ems, avg_min_ems)

                if workflow in reduction_by_workflow:
                    reduction_by_workflow[workflow].append(avg_reduction)
                else:
                    reduction_by_workflow[workflow] = [avg_reduction]
    
        data[rep] = reduction_by_workflow

    return data

In [55]:
# Sensitivity Analysis -- CI

# Sensitivity Analysis -- Runtime

# Sensitivity Analysis -- Runtime + CI

# Upper + Lower Bound for Storage Costs during interrupted workflow shifting (scenario where we extend time spent)

In [ ]:
## Sensitivity Analysis - CI Experiments
region = 'gb'

# Average CI (24h)
read_24h = parse_readings(window=24)
curr = []
for workflow in workflows:
    curr.append(read_24h[workflow])
reads_24h_original = np.array(curr)

reads_24h_err5 = parse_readings(window=24, err='-err5')
all_reads_24h_err5 = []
for iter in range(1, 6):
    curr = []
    for workflow in workflows:
        curr.append(reads_24h_err5[iter][workflow])
    all_reads_24h_err5.append(np.array(curr))
reads_24h_err5_mean = np.mean(all_reads_24h_err5, axis=0)

# reads_24h_err10 = parse_readings(window=24, err='-err10')
# all_reads_24h_err10 = []
# for iter in range(1, 6):
#     curr = []
#     for workflow in workflows:
#         curr.append(reads_24h_err10[iter][workflow])
#     all_reads_24h_err10.append(np.array(curr))
# reads_24h_err10_mean = np.mean(all_reads_24h_err10, axis=0)

# # 5% error with 24h readings
# score = r2_score(reads_24h_original, test_arr)
# print(score)

# # 10% error with 96h readings
# score = r2_score(reads_24h_original, test_arr)
# print(score)



FileNotFoundError: [Errno 2] No such file or directory: '../data/results/entire-wf-shifting/out-err/gb/explorer-24h-chipseq-jan-1-gb-08012024-13012024err5~footprint.csv'

In [ ]:
# not 100% sure if it makes sense to blindly adjust timestamps of a workflow to make tasks have 'inaccurate predictions'ArithmeticError
# thinking about having some illustrative examples, if we imagine that we have a prediction error of X
# workflow could take Y longer, which leads to footprint increase by Z
# any method is reliant on having reliable predictions of runtime 

# will try adding noise but i dont know if it makes sense at all